In [648]:
import pandas as pd
import re
import plotly.graph_objects as go
import json

In [649]:
raw_ensg_df = pd.read_csv("../output/raw_ensg_df.csv")
raw_hgnc_df = pd.read_csv("../output/raw_hgnc_df.csv")
raw_ncbi_df = pd.read_csv("../output/raw_ncbi_df.csv")
ensg_gene_type_df = pd.read_csv("../input/ensg_gene_type202508.txt")
hgnc_gene_type_df = pd.read_csv("../input/hgnc_gene_type_20250813.txt", sep="\t")

# Gene Record Type

In [650]:
ensg_gene_type_df = ensg_gene_type_df.rename(columns={"Gene stable ID": "ENSG_ID", "Gene name": "gene_symbol","Gene type":"gene_type"})

In [651]:
ensg_df = raw_ensg_df.merge(ensg_gene_type_df[["ENSG_ID", "gene_type"]], on="ENSG_ID", how="left")
ensg_df

,ENSG_ID,NCBI_ID,HGNC_ID,alias_symbol,gene_symbol,gene_type
0,ENSG00000210049,NaN,HGNC:7481,MTTF,MT-TF,Mt_tRNA
1,ENSG00000210049,NaN,HGNC:7481,TRNF,MT-TF,Mt_tRNA
2,ENSG00000211459,NaN,HGNC:7470,12S,MT-RNR1,Mt_rRNA
3,ENSG00000211459,NaN,HGNC:7470,MOTS-C,MT-RNR1,Mt_rRNA
4,ENSG00000211459,NaN,HGNC:7470,MTRNR1,MT-RNR1,Mt_rRNA
...,...,...,...,...,...,...
133060,ENSG00000229388,NaN,HGNC:52502,LINC01715,TAF12-DT,lncRNA
133061,ENSG00000289291,NaN,NaN,NaN,NaN,lncRNA
133062,ENSG00000274978,GENE ID:26824,HGNC:10108,RNU11-1,RNU11,snRNA
133063,ENSG00000274978,GENE ID:26824,HGNC:10108,U11,RNU11,snRNA


In [652]:
ensg_df["gene_type"].value_counts()

gene_type
protein_coding                        57993
lncRNA                                38719
processed_pseudogene                  11136
rRNA                                   4032
unprocessed_pseudogene                 3956
transcribed_unprocessed_pseudogene     3003
miRNA                                  2844
snRNA                                  2664
misc_RNA                               2437
transcribed_processed_pseudogene       1425
snoRNA                                 1241
TEC                                    1090
rRNA_pseudogene                         525
TR_V_gene                               368
IG_V_pseudogene                         318
transcribed_unitary_pseudogene          315
IG_V_gene                               262
TR_J_gene                               124
unitary_pseudogene                      118
TR_V_pseudogene                         108
IG_D_gene                               102
scaRNA                                   56
Mt_tRNA               

https://useast.ensembl.org/info/genome/genebuild/biotypes.html

https://www.ncbi.nlm.nih.gov/books/NBK3841/#EntrezGene.Properties

no readthrough? even though in biotype doc

In [653]:
ensg_gene_type_value_map = {
    "protein_coding":"Protein Coding",
    "lncRNA": "Non Coding",
    "processed_pseudogene": "Pseudogene",
    "rRNA": "Non Coding",
    "unprocessed_pseudogene": "Pseudogene",
    "transcribed_unprocessed_pseudogene": "Pseudogene",
    "miRNA": "Non Coding",
    "snRNA": "Non Coding",
    "misc_RNA": "Non Coding",
    "transcribed_processed_pseudogene": "Pseudogene",
    "snoRNA": "Non Coding",
    "TEC": "Unknown",
    "rRNA_pseudogene": "Pseudogene",
    "TR_V_gene": "Other",
    "IG_V_pseudogene": "Pseudogene",
    "transcribed_unitary_pseudogene": "Pseudogene",
    "IG_V_gene": "Other",
    "TR_J_gene": "Other",
    "unitary_pseudogene": "Pseudogene",
    "TR_V_pseudogene": "Pseudogene",
    "IG_D_gene": "Other",
    "scaRNA": "Non Coding",
    "Mt_tRNA": "Non Coding",
    "IG_C_gene": "Other",
    "IG_J_gene": "Other",
    "vault_RNA": "Non Coding",
    "pseudogene": "Pseudogene",
    "artifact": "Other",
    "TR_C_gene": "Other",
    "ribozyme": "Non Coding",
    "IG_C_pseudogene": "Pseudogene",
    "TR_D_gene": "Other",
    "sRNA": "Non Coding",
    "Mt_rRNA": "Non Coding",
    "IG_J_pseudogene": "Pseudogene",
    "TR_J_pseudogene": "Pseudogene",
    "translated_processed_pseudogene": "Pseudogene",
    "IG_pseudogene":"Pseudogene"
}

In [654]:
ensg_df["simplified_gene_type"] = ensg_df["gene_type"].map(ensg_gene_type_value_map)

In [655]:
hgnc_gene_type_df = hgnc_gene_type_df.rename(columns={"HGNC ID":"HGNC_ID","Approved symbol":"gene_symbol","Locus type":"gene_type" })

In [656]:
hgnc_df = raw_hgnc_df.merge(hgnc_gene_type_df[["HGNC_ID", "gene_type"]], on="HGNC_ID", how="left")
hgnc_df

,HGNC_ID,gene_symbol,alias_symbol,NCBI_ID,ENSG_ID,gene_type
0,HGNC:100,ASIC1,BNaC2,GENE ID:41,ENSG00000110881,gene with protein product
1,HGNC:100,ASIC1,hBNaC2,GENE ID:41,ENSG00000110881,gene with protein product
2,HGNC:10000,RGS4,NaN,GENE ID:5999,ENSG00000117152,gene with protein product
3,HGNC:10001,RGS5,NaN,GENE ID:8490,ENSG00000143248,gene with protein product
4,HGNC:10002,RGS6,NaN,GENE ID:9628,ENSG00000182732,gene with protein product
...,...,...,...,...,...,...
66202,HGNC:9997,RGS16,RGS-r,GENE ID:6004,ENSG00000143333,gene with protein product
66203,HGNC:9998,RGS2,NaN,GENE ID:5997,ENSG00000116741,gene with protein product
66204,HGNC:9999,RGS3,C2PA,GENE ID:5998,ENSG00000138835,gene with protein product
66205,HGNC:9999,RGS3,FLJ20370,GENE ID:5998,ENSG00000138835,gene with protein product


In [657]:
hgnc_df["gene_type"].value_counts()

gene_type
gene with protein product     39493
pseudogene                    14725
RNA, long non-coding           6850
RNA, micro                     1979
RNA, small nucleolar            649
RNA, transfer                   598
T cell receptor gene            348
immunoglobulin gene             276
immunoglobulin pseudogene       208
readthrough                     161
endogenous retrovirus           132
RNA, cluster                    119
fragile site                    116
complex locus constituent       114
RNA, small nuclear              102
unknown                          91
T cell receptor pseudogene       73
RNA, ribosomal                   62
region                           48
RNA, misc                        32
RNA, vault                       16
virus integration site            9
RNA, Y                            4
Name: count, dtype: int64

https://www.genenames.org/help/symbol-report/

In [658]:
hgnc_gene_type_value_map = {
    "gene with protein product":"Protein Coding",
    "RNA, long non-coding": "Non Coding",
    "pseudogene": "Pseudogene",
    "RNA, micro": "Non Coding",
    "RNA, small nucleolar": "Non Coding",
    "RNA, transfer": "Non Coding",
    "T cell receptor gene": "Other",
    "immunoglobulin gene": "Other",
    "immunoglobulin pseudogene": "Pseudogene",
    "readthrough": "Readthrough",
    "endogenous retrovirus": "Other",
    "RNA, cluster": "Non Coding",
    "fragile site": "Other",
    "complex locus constituent": "Other",
    "RNA, small nuclear": "Non Coding",
    "unknown": "Unknown",
    "T cell receptor pseudogene": "Pseudogene",
    "RNA, ribosomal": "Non Coding",
    "region": "Other",
    "RNA, misc": "Non Coding",
    "RNA, vault": "Non Coding",
    "virus integration site ": "Other",
    "RNA, Y ": "Non Coding"
}

In [659]:
hgnc_df["simplified_gene_type"] = hgnc_df["gene_type"].map(hgnc_gene_type_value_map)

In [660]:
ncbi_df = raw_ncbi_df.copy()

In [661]:
ncbi_df["gene_type"].value_counts()

gene_type
biological-region    128312
protein-coding        60978
ncRNA                 24899
pseudo                20322
snoRNA                 1412
other                  1312
unknown                1297
tRNA                    894
rRNA                    791
snRNA                   279
scRNA                    12
Name: count, dtype: int64

In [662]:
ncbi_gene_type_value_map = {
    "biological-region":"Non Coding",
    "protein-coding": "Protein Coding",
    "ncRNA": "Non Coding",
    "pseudo": "Pseudogene",
    "snoRNA": "Non Coding",
    "other": "Other",
    "unknown": "Unknown",
    "tRNA": "Non Coding",
    "rRNA": "Non Coding",
    "snRNA": "Non Coding",
    "scRNA": "Non Coding"
}

In [663]:
ncbi_df["simplified_gene_type"] = ncbi_df["gene_type"].map(ncbi_gene_type_value_map)

# Primary Symbol Syntax

In [664]:
with open('../output/pheno_records_in_ncbi_set.json', 'r') as f:
    pheno_records_in_ncbi_set = set(json.load(f))

In [665]:
with open('../output/hum_subspec_records_in_ncbi_set.json', 'r') as f:
    hum_subspec_records_in_ncbi_set = set(json.load(f))

In [666]:
def classify_criteria(symbol,record_id=None):
    if pd.isna(symbol):
        return "No Primary Symbol"

    if symbol.startswith("KIAA") and "-" not in symbol:
        return "KIAA"
    
    if symbol.startswith("LOC"):
        return "LOC"
    
    if re.match(r"^C.*ORF", symbol, re.IGNORECASE) and "-" not in symbol:
        return "C-orf"
    
    if symbol.startswith("FAM") and "-" not in symbol:
        return "FAM"
    
    if "@" in symbol:
        return "@ symbol"
    
    if record_id is not None:
            if record_id in hum_subspec_records_in_ncbi_set:
                return "Human Sub-Species"
            
            if record_id in pheno_records_in_ncbi_set:
                return "Phenotype"

    return "other"


In [667]:
ensg_df['gene_symbol_type'] = ensg_df['gene_symbol'].apply(classify_criteria)
hgnc_df['gene_symbol_type'] = hgnc_df['gene_symbol'].apply(classify_criteria)
ncbi_df['gene_symbol_type'] = ncbi_df.apply(lambda row: classify_criteria(row['gene_symbol'], row['NCBI_ID']), axis=1)

# Data Retention

In [668]:
mini_ensg_df = pd.read_csv("../output/mini_ensg_df.csv")
mini_hgnc_df = pd.read_csv("../output/mini_hgnc_df.csv")
mini_ncbi_df = pd.read_csv("../output/mini_ncbi_df.csv")

In [669]:
ensg_df["data_retention"] = ensg_df["ENSG_ID"].isin(mini_ensg_df["ENSG_ID"])
hgnc_df["data_retention"] = hgnc_df["HGNC_ID"].isin(mini_hgnc_df["HGNC_ID"])
ncbi_df["data_retention"] = ncbi_df["NCBI_ID"].isin(mini_ncbi_df["NCBI_ID"])

# Database allocation

In [670]:
ensg_df["database"] = "ENSG"
hgnc_df["database"] = "HGNC"
ncbi_df["database"] = "NCBI"

In [671]:
sub = ncbi_df.loc[ncbi_df["gene_symbol_type"] == "LOC"]
sub["simplified_gene_type"].value_counts()

simplified_gene_type
Non Coding        143704
Pseudogene          2904
Protein Coding      1160
Other                 18
Unknown                3
Name: count, dtype: int64

# making a table for the sankey

In [672]:
abbrev_ensg_df = ensg_df[["ENSG_ID", "NCBI_ID", "HGNC_ID","simplified_gene_type","gene_symbol_type","data_retention","database"]]
abbrev_hgnc_df = hgnc_df[["ENSG_ID", "NCBI_ID", "HGNC_ID","simplified_gene_type","gene_symbol_type","data_retention","database"]]
abbrev_ncbi_df = ncbi_df[["ENSG_ID", "NCBI_ID", "HGNC_ID","simplified_gene_type","gene_symbol_type","data_retention","database"]]

In [673]:
combined_df = pd.concat([abbrev_ensg_df, abbrev_hgnc_df, abbrev_ncbi_df], ignore_index=True)

In [674]:
combined_df["all_records"] = "All Records"

### version 1

In [675]:
cols = ["all_records", "simplified_gene_type", "database", "gene_symbol_type", "data_retention"]

# Create a list of all unique values across all columns (used as nodes)
labels = pd.unique(combined_df[cols].values.ravel()).tolist()

# Create a mapping from each label to an index (needed for Sankey)
label_to_index = {label: i for i, label in enumerate(labels)}

# Create empty lists to store Sankey source, target, and value
sources = []
targets = []
values = []

# Loop through adjacent column pairs to create links
for i in range(len(cols) - 1):
    col1 = cols[i]
    col2 = cols[i + 1]
    
    # Group by adjacent columns and count the occurrences
    grouped = combined_df.groupby([col1, col2]).size().reset_index(name='count')
    
    # Add the index of the source and target labels to the lists
    for _, row in grouped.iterrows():
        src_label = row[col1]
        tgt_label = row[col2]
        count = row['count']
        
        sources.append(label_to_index[src_label])
        targets.append(label_to_index[tgt_label])
        values.append(count)


In [676]:
fig = go.Figure(data=[go.Sankey(
    arrangement="snap",  # optional: controls node layout
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=labels,  # the unique values from all columns
    ),
    link=dict(
        source=sources,  # indices in 'labels' list
        target=targets,
        value=values
    )
)])

fig.update_layout(title_text="Analysis Subset Pruning Diagram", font_size=12)
fig.show()


### version 2

In [677]:
v2_combined_df = combined_df.drop('simplified_gene_type', axis=1)

In [678]:
v2_combined_df

,ENSG_ID,NCBI_ID,HGNC_ID,gene_symbol_type,data_retention,database,all_records
0,ENSG00000210049,NaN,HGNC:7481,other,True,ENSG,All Records
1,ENSG00000210049,NaN,HGNC:7481,other,True,ENSG,All Records
2,ENSG00000211459,NaN,HGNC:7470,other,True,ENSG,All Records
3,ENSG00000211459,NaN,HGNC:7470,other,True,ENSG,All Records
4,ENSG00000211459,NaN,HGNC:7470,other,True,ENSG,All Records
...,...,...,...,...,...,...,...
439775,NaN,GENE ID:8923215,NaN,Human Sub-Species,False,NCBI,All Records
439776,NaN,GENE ID:8923216,NaN,Human Sub-Species,False,NCBI,All Records
439777,NaN,GENE ID:8923217,NaN,Human Sub-Species,False,NCBI,All Records
439778,NaN,GENE ID:8923218,NaN,Human Sub-Species,False,NCBI,All Records


In [679]:
cols = ["all_records", "database", "gene_symbol_type", "data_retention"]

# Create a list of all unique values across all columns (used as nodes)
labels = pd.unique(v2_combined_df[cols].values.ravel()).tolist()

# Create a mapping from each label to an index (needed for Sankey)
label_to_index = {label: i for i, label in enumerate(labels)}

# Create empty lists to store Sankey source, target, and value
sources = []
targets = []
values = []

# Loop through adjacent column pairs to create links
for i in range(len(cols) - 1):
    col1 = cols[i]
    col2 = cols[i + 1]
    
    # Group by adjacent columns and count the occurrences
    grouped = v2_combined_df.groupby([col1, col2]).size().reset_index(name='count')
    
    # Add the index of the source and target labels to the lists
    for _, row in grouped.iterrows():
        src_label = row[col1]
        tgt_label = row[col2]
        count = row['count']
        
        sources.append(label_to_index[src_label])
        targets.append(label_to_index[tgt_label])
        values.append(count)


In [680]:
fig = go.Figure(data=[go.Sankey(
    arrangement="snap",  # optional: controls node layout
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=labels,  # the unique values from all columns
    ),
    link=dict(
        source=sources,  # indices in 'labels' list
        target=targets,
        value=values
    )
)])
# Add custom stage labels (these correspond to your 'cols')
stage_labels = ["", "Database", "Exclusion Criteria", "Analysis Subset"]
num_stages = len(stage_labels)

# Estimate x positions for each stage (spread evenly)
x_positions = [i / (num_stages - 1) for i in range(num_stages)]

# Manual adjustments
x_positions[2] -= 0.03  # shift "Gene Symbol Type" left
x_positions[3] += 0.03  # shift "Analysis Subset" right

label_y_pos = 1.09

for i, label in enumerate(stage_labels):
    if label:  # skip empty labels if any
        fig.add_annotation(
            x=x_positions[i],
            y=label_y_pos,
            text=label,
            showarrow=False,
            font=dict(size=14, color="black"),
            xref="paper",
            yref="paper"
        )

fig.update_layout(title_text="Analysis Subset Pruning Diagram", font_size=12)
fig.show()